# DBRepo Upload Notebook

This notebook handles the creation of the database, tables, and the upload of the data to DBRepo via the REST API. It ensures proper metadata attribution to Eurostat and applies the CC BY 4.0 license.

**Note:** As per the plan, this notebook sets up the logic but data changes should only be executed once fully approved.

In [3]:
import requests
import pandas as pd
import json

DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
DATABASE_ID = "123289f2-5218-4b32-b962-5f3dafec1fe3"
USERNAME = ""
PASSWORD = ""

In [4]:
# Metadata for the Database
database_metadata = {
    "name": "EU_Environmental_Investment_Analysis",
    "description": "Environmental protection investments across EU countries (2014-2022).",
    "publisher": "Eurostat",
    "creator": "Eurostat",
    "license": "CC BY 4.0",
    "rights": "European Union / Eurostat",
    "republisher": "Daniel Wrnr / TU Wien"
}
print("Database metadata prepared:", json.dumps(database_metadata, indent=2))

Database metadata prepared: {
  "name": "EU_Environmental_Investment_Analysis",
  "description": "Environmental protection investments across EU countries (2014-2022).",
  "publisher": "Eurostat",
  "creator": "Eurostat",
  "license": "CC BY 4.0",
  "rights": "European Union / Eurostat",
  "republisher": "Daniel Wrnr / TU Wien"
}


## Create Tables via REST API

Here we define the schemas for the four tables: `Country`, `Environmental_Activity`, `Macroeconomic_Indicator`, and `Environmental_Investment`. We use the `requests` library to create them directly via the DBRepo REST API to include primary keys, foreign keys, and descriptive metadata.

In [5]:


table_country = {
    "name": "Country",
    "is_public": True,
    "is_schema_public": True,
    "description": "Country dimension table with ISO codes.",
    "columns": [
        {"name": "country_code", "type": "varchar", "size": 2, "null_allowed": False, "description": "2-letter ISO country code"},
        {"name": "country_name", "type": "varchar", "size": 255, "null_allowed": False, "description": "Full name of the country"}
    ],
    "constraints": {
        "primary_key": ["country_code"]
    }
}

table_activity = {
    "name": "Environmental_Activity",
    "is_public": True,
    "is_schema_public": True,
    "description": "Environmental activity dimension table (CEPA/CReMA classifications).",
    "columns": [
        {"name": "ceparema_code", "type": "varchar", "size": 50, "null_allowed": False, "description": "CEPA/CReMA activity code"},
        {"name": "activity_name", "type": "varchar", "size": 255, "null_allowed": False, "description": "Name of the environmental protection activity"}
    ],
    "constraints": {
        "primary_key": ["ceparema_code"]
    }
}


In [6]:
table_macro = {
    "name": "Macroeconomic_Indicator",
    "is_public": True,
    "is_schema_public": True,
    "columns": [
        {"name": "country_code", "type": "varchar", "size": 2, "null_allowed": False, "description": "2-letter ISO country code"},
        {"name": "year", "type": "int", "null_allowed": False, "description": "Observation year"},
        {"name": "population", "type": "bigint", "null_allowed": True, "description": "Total population"},
        {"name": "gdp_per_capita", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Gross Domestic Product per capita"}
    ],
    "constraints": {
        "primary_key": ["country_code", "year"],
        "foreign_keys": [
            {
                "columns": ["country_code"],
                "referenced_table": "country",
                "referenced_columns": ["country_code"]
            }
        ]
    }
}

table_invest = {
    "name": "Environmental_Investment",
    "is_public": True,
    "is_schema_public": True,
    "columns": [
        {"name": "country_code", "type": "varchar", "size": 2, "null_allowed": False, "description": "2-letter ISO country code"},
        {"name": "year", "type": "int", "null_allowed": False, "description": "Observation year"},
        {"name": "ceparema_code", "type": "varchar", "size": 50, "null_allowed": False, "description": "CEPA/CReMA activity code"},
        {"name": "inv_gov", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Investment by general government"},
        {"name": "inv_corp_spec", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Investment by specialist producers"},
        {"name": "inv_corp_anc", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Investment by ancillary producers"},
        {"name": "inv_corp_total", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Total corporate investment"},
        {"name": "inv_total", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Total environmental investment"}
    ],
    "constraints": {
        "primary_key": ["country_code", "year", "ceparema_code"],
        "foreign_keys": [
            {
                "columns": ["country_code"],
                "referenced_table": "country",
                "referenced_columns": ["country_code"]
            },
            {
                "columns": ["ceparema_code"],
                "referenced_table": "environmental_activity",
                "referenced_columns": ["ceparema_code"]
            }
        ]
    }
}


In [ ]:
def create_table_via_api(database_id, table_def):
    # Ensure all constraint fields are present to avoid 500 errors
    if "constraints" not in table_def:
        table_def["constraints"] = {}
    
    for key in ["uniques", "checks", "foreign_keys", "primary_key"]:
        if key not in table_def["constraints"]:
            table_def["constraints"][key] = []

    url = f"{DBREPO_ENDPOINT}/api/v1/database/{database_id}/table"
    print(f"Creating table {table_def['name']}...")
    response = requests.post(url, json=table_def, auth=(USERNAME, PASSWORD))
    
    if response.status_code == 201:
        print(f"Success: Table {table_def['name']} created.")
    elif response.status_code == 409:
        print(f"Warning: Table {table_def['name']} already exists.")
    else:
        print(f"Error ({response.status_code}): {response.text}")


tables_to_create = [table_country, table_activity, table_macro, table_invest]
for t in tables_to_create:
    create_table_via_api(DATABASE_ID, t)


Creating table Country...
Creating table Environmental_Activity...
Creating table Macroeconomic_Indicator...
Creating table Environmental_Investment...
